# Stock Price Time Series Analysis

Download five years of adjusted closing prices and daily share volume for Apple (`AAPL`) and Microsoft (`MSFT`). This notebook demonstrates date indexes, line charts, weekly/monthly resampling, rolling averages, and a volume comparison. It downloads the market data when the first code cell runs.

## 1. Download and index the daily history

`yfinance.download` uses an inclusive start date and exclusive end date. The price frame keeps the trading date as a `DatetimeIndex`; the adjusted close accounts for dividends and splits.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from stock_analysis import (
    TICKERS, calculate_moving_averages, compare_trading_volume,
    download_market_data, resample_prices, save_results,
)

prices, volume = download_market_data()
print(f'{len(prices):,} trading days: {prices.index.min():%Y-%m-%d} to {prices.index.max():%Y-%m-%d}')
prices.index

## 2. Plot adjusted closing prices

In [ ]:
ax = prices.plot(figsize=(12, 6), linewidth=1.4)
ax.set(title='Adjusted Closing Prices', xlabel='Date', ylabel='Adjusted close (USD)')
ax.grid(alpha=0.25)
ax.legend(title='Ticker')
plt.tight_layout()
plt.show()

## 3. Resample to weekly and monthly frequencies

Each period uses the last available trading-day close. This avoids treating weekends and market holidays as observations.

In [ ]:
weekly, monthly = resample_prices(prices)
display(weekly.head())
display(monthly.head())
print(f'Weekly rows: {len(weekly):,} | Monthly rows: {len(monthly):,}')

## 4. Calculate and plot the 20-day and 200-day moving averages

The windows count trading sessions, not calendar days; the initial rows are missing until a complete window is available.

In [ ]:
ma20, ma200 = calculate_moving_averages(prices)
fig, axes = plt.subplots(len(TICKERS), 1, figsize=(12, 5 * len(TICKERS)), sharex=True)
for ax, ticker in zip(axes, TICKERS):
    ax.plot(prices.index, prices[ticker], label='Adjusted Close', color='#3259a8', linewidth=1.1)
    ax.plot(ma20.index, ma20[ticker], label='20-day average', color='#f39c12', linewidth=1.2)
    ax.plot(ma200.index, ma200[ticker], label='200-day average', color='#c0392b', linewidth=1.4)
    ax.set_title(f'{ticker}: Price and Moving Averages')
    ax.set_ylabel('USD')
    ax.grid(alpha=0.2)
    ax.legend()
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

## 5. Compare daily trading volume

Compare the mean number of shares traded per session over the selected five-year window.

In [ ]:
average_volume = compare_trading_volume(volume)
display(average_volume.map(lambda value: f'{value:,.0f}').rename('Average Shares / Day').to_frame())
ax = average_volume.sort_values().plot.bar(figsize=(8, 5), color=['#168aad', '#f4a261'])
ax.set(title='Average Daily Trading Volume', xlabel='Ticker', ylabel='Shares traded per day')
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Save tables and charts

The same outputs can be generated from the command line with `python stock_analysis.py`.

In [ ]:
save_results(prices, volume, weekly, monthly, ma20, ma200)
print('Saved price/volume CSV files under data/ and summaries/charts under outputs/.')